In [ ]:
# === auto-inserted: bev-solving src on path ===
import sys, pathlib
_root = pathlib.Path.cwd()
while _root != _root.parent and not (_root / 'src' / 'geometry.py').exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


# Train BEV v4 RTMDet Pretrain

Self-contained notebook for camera-only BEV training with:
- merged `train + val` pool;
- empty-GT removal;
- smart dedup of near-static frames;
- test-matched grouped validation around 200 samples;
- compact rover embeddings with `Other=0`;
- RTMDet-L pretrained `CSPNeXt` camera encoder loaded directly from a `.pth` checkpoint;
- optional 2-GPU training via `nn.DataParallel`.

This notebook does not modify the source dataset folders. All caches and checkpoints are written only into `RUN_DIR`.


In [ ]:
%load_ext autoreload
%autoreload 2

import os, copy, time, json, random, gc
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageFile
from tqdm import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

DATA_TRAIN = Path('./autonomy_yandex_dataset_train/')
DATA_VAL   = Path('./autonomy_yandex_dataset_val/')
DATA_TEST  = Path('./autonomy_yandex_dataset_test/')
RTMDET_PRETRAIN_PATH = Path('./rtmdet_l_8xb32-300e_coco_20220719_112030-5a0be7c4.pth')

cfg = {
    'run_dir': './runs/v4_rtmdet_pretrain',
    'img_hw': (384, 704),
    'batch_size': 8,
    'val_batch_size': 1,
    'grad_accum': 4,
    'epochs': 20,
    'warmup_epochs': 3,
    'lr_backbone': 3e-5,
    'lr_head': 3e-4,
    'weight_decay': 1e-4,
    'embed_weight_decay': 1e-2,
    'pos_weight': 5.0,
    'num_workers': 4,
    'val_num_workers': 0,
    'seed': 42,
    'ema_decay': 0.995,
    'val_target_size': 200,
    'min_rover_count': 30,
    'topk_rovers': 25,
    'rover_emb_dim': 8,
    'rover_cond_dim': 8,
    'mae_dedup_thr': 0.02,
    'dedup_camera': '/camera/inner/frontal/middle',
    'use_clean_cache': True,
    'use_dp_if_available': True,
    'pretrained_backbone_path': str(RTMDET_PRETRAIN_PATH),
}

RUN_DIR = Path(cfg['run_dir'])
RUN_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = RUN_DIR / 'preproc_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

random.seed(cfg['seed'])
np.random.seed(cfg['seed'])
torch.manual_seed(cfg['seed'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_cuda = torch.cuda.device_count() if torch.cuda.is_available() else 0
print('device:', device)
print('cuda devices:', num_cuda)
if num_cuda:
    for i in range(num_cuda):
        print(i, torch.cuda.get_device_name(i))
print('pretrain path:', RTMDET_PRETRAIN_PATH)
print('pretrain exists:', RTMDET_PRETRAIN_PATH.exists())
print('train batch_size:', cfg['batch_size'], '| train workers:', cfg['num_workers'])
print('val batch_size:', cfg['val_batch_size'], '| val workers:', cfg['val_num_workers'])

with open(RUN_DIR / 'config.json', 'w') as f:
    json.dump({k: str(v) for k, v in cfg.items()}, f, indent=2)



In [ ]:
CAMERA_NAMES = [
    '/camera/inner/frontal/middle',
    '/camera/inner/frontal/far',
    '/side/left/forward',
    '/side/right/forward',
]
INTRINSICS_NAMES = [n + '/intrinsic_params' for n in CAMERA_NAMES]
CAR2CAM_NAMES = [n + '/car_to_cam' for n in CAMERA_NAMES]
GT_NAME = 'gt_occupancy_grid'

BEV_H, BEV_W = 188, 126
BEV_RES = 0.8
X_RANGE = (0.0, BEV_H * BEV_RES)
Y_RANGE = (-BEV_W * BEV_RES / 2, BEV_W * BEV_RES / 2)
Z_LEVELS = (0.3, 1.0, 2.0, 3.0)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def resolve_info_path(base_dir: Path, p) -> Path:
    p = Path(p)
    if p.is_absolute() and p.exists():
        return p
    if p.exists():
        return p
    cand = base_dir / p
    if cand.exists():
        return cand
    cand = base_dir.parent / p
    if cand.exists():
        return cand
    return base_dir.parent / p


def load_info_with_root(data_dir: Path, split_name: str) -> pd.DataFrame:
    df = pd.read_csv(data_dir / 'info.csv', index_col=0).reset_index(drop=True).copy()
    df['__data_root'] = str(data_dir)
    df['__source_split'] = split_name
    return df


def resolve_row_path(row: pd.Series, key: str) -> Path:
    return resolve_info_path(Path(row['__data_root']), row[key])


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.cleaning import build_img_hash, clean_merged_info, compute_gt_stats, smart_deduplicate


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.splits import build_rover_vocab_from_train, encode_rover, make_test_matched_split_target


In [ ]:
clean_info, dedup_report, clean_summary = clean_merged_info(
    DATA_TRAIN,
    DATA_VAL,
    cache_dir=CACHE_DIR,
    mae_thr=cfg['mae_dedup_thr'],
    dedup_camera=cfg['dedup_camera'],
    use_cache=cfg['use_clean_cache'],
)
print(json.dumps(clean_summary, indent=2, ensure_ascii=False))
display(clean_info.head(3))
if len(dedup_report):
    display(dedup_report.head(10))

train_idx, val_idx = make_test_matched_split_target(
    clean_info,
    DATA_TEST / 'info.csv',
    target_val_size=cfg['val_target_size'],
    seed=cfg['seed'],
    cache_path=CACHE_DIR / 'test_matched_val200_split.npz',
)
print('train_idx:', len(train_idx), 'val_idx:', len(val_idx))

train_info = clean_info.iloc[train_idx].reset_index(drop=True).copy()
val_info = clean_info.iloc[val_idx].reset_index(drop=True).copy()

rover_vocab, rover_stats = build_rover_vocab_from_train(
    train_info,
    min_count=cfg['min_rover_count'],
    topk=cfg['topk_rovers'],
)
rover_stats.to_csv(RUN_DIR / 'rover_embedding_stats.csv', index=False)
print('num rover classes including Other:', len(rover_vocab))
print('vocab head:', list(rover_vocab.items())[:10])
display(rover_stats.head(30))


In [ ]:
class BEVDatasetV4RTMDet(Dataset):
    def __init__(self, info_df: pd.DataFrame, mode: str = 'train',
                 img_hw=(384, 704), aug: bool = False,
                 rover_vocab: dict | None = None):
        self.info = info_df.reset_index(drop=True).copy()
        self.mode = mode
        self.img_hw = img_hw
        self.aug = aug and (mode == 'train')
        self.rover_vocab = rover_vocab or {'__other__': 0}
        self.normalize = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)

    def __len__(self):
        return len(self.info)

    def _resolve_path(self, row: pd.Series, key: str) -> Path:
        return resolve_info_path(Path(row['__data_root']), row[key])

    def _load_one_camera(self, row: pd.Series, cam_key: str, intr_key: str, c2c_key: str,
                         scale_aug: float = 1.0):
        img = Image.open(self._resolve_path(row, cam_key)).convert('RGB')
        intr_path = self._resolve_path(row, intr_key)
        car2cam_path = self._resolve_path(row, c2c_key)
        src_W, src_H = img.size
        H_t, W_t = self.img_hw

        s = min(W_t / src_W, H_t / src_H)
        new_W, new_H = int(round(src_W * s)), int(round(src_H * s))
        img_resized = img.resize((new_W, new_H), Image.BILINEAR)
        canvas = Image.new('RGB', (W_t, H_t), (0, 0, 0))
        pad_x = (W_t - new_W) // 2
        pad_y = (H_t - new_H) // 2
        canvas.paste(img_resized, (pad_x, pad_y))

        extra_s = 1.0
        extra_dx = 0
        extra_dy = 0
        if scale_aug > 1.0:
            sH, sW = int(round(H_t * scale_aug)), int(round(W_t * scale_aug))
            canvas = canvas.resize((sW, sH), Image.BILINEAR)
            extra_dx = random.randint(0, sW - W_t)
            extra_dy = random.randint(0, sH - H_t)
            canvas = canvas.crop((extra_dx, extra_dy, extra_dx + W_t, extra_dy + H_t))
            extra_s = scale_aug

        arr = np.array(canvas)
        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)
        img_t = torch.from_numpy(arr).permute(2, 0, 1).float() / 255.0
        img_t = self.normalize(img_t)

        intr_full = np.load(intr_path)
        K = intr_full[:, :3].copy().astype(np.float32)
        K[0, 0] *= s; K[0, 2] *= s
        K[1, 1] *= s; K[1, 2] *= s
        K[0, 2] += pad_x
        K[1, 2] += pad_y
        K[0, 0] *= extra_s; K[0, 2] *= extra_s
        K[1, 1] *= extra_s; K[1, 2] *= extra_s
        K[0, 2] -= extra_dx
        K[1, 2] -= extra_dy

        car2cam = np.load(car2cam_path).astype(np.float32)
        return img_t, K, car2cam

    def _load_sample(self, idx: int):
        row = self.info.iloc[idx]
        scale_aug = random.uniform(1.0, 1.15) if self.aug else 1.0

        imgs, Ks, M = [], [], []
        for cam, intr, c2c in zip(CAMERA_NAMES, INTRINSICS_NAMES, CAR2CAM_NAMES):
            img_t, K, c = self._load_one_camera(row, cam, intr, c2c, scale_aug=scale_aug)
            imgs.append(img_t)
            Ks.append(torch.from_numpy(K))
            M.append(torch.from_numpy(c))

        out = {
            'images': torch.stack(imgs, dim=0),
            'intrinsics': torch.stack(Ks, dim=0),
            'car2cams': torch.stack(M, dim=0),
            'rover_id': torch.tensor(encode_rover(row.get('rover', '__other__'), self.rover_vocab), dtype=torch.long),
            'info_idx': idx,
        }
        if self.mode == 'test':
            return out
        gt = np.load(self._resolve_path(row, GT_NAME)).squeeze()
        gt = np.where(gt < 0, 255, gt).astype(np.int64)
        out['gt'] = torch.from_numpy(gt).unsqueeze(0)
        return out

    def __getitem__(self, idx: int):
        max_tries = 5
        last_err = None
        for k in range(max_tries):
            try_idx = (idx + k) % len(self.info)
            try:
                return self._load_sample(try_idx)
            except (OSError, ValueError, FileNotFoundError) as e:
                last_err = e
                continue
        raise RuntimeError(f'Failed to load sample after {max_tries} tries from idx={idx}: {last_err}')


In [ ]:
class ConvBNAct(nn.Module):
    def __init__(self, in_c, out_c, k, stride=1, padding=None, groups=1,
                 bias=False, eps=1e-3, momentum=0.01, act=True):
        super().__init__()
        if padding is None:
            padding = k // 2
        self.conv = nn.Conv2d(in_c, out_c, k, stride=stride, padding=padding, groups=groups, bias=bias)
        self.bn = nn.BatchNorm2d(out_c, eps=eps, momentum=momentum)
        self.act = nn.SiLU(inplace=True) if act else nn.Identity()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class ChannelAttention(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Conv2d(channels, channels, 1, 1, 0, bias=True)
        self.act = nn.Hardsigmoid(inplace=True)

    def forward(self, x):
        with torch.cuda.amp.autocast(enabled=False):
            a = self.pool(x.float())
            a = self.fc(a)
            a = self.act(a)
        return x * a.to(dtype=x.dtype)


class SPPBottleneck(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_sizes=(5, 9, 13), eps=1e-3, momentum=0.01):
        super().__init__()
        mid_channels = in_channels // 2
        self.conv1 = ConvBNAct(in_channels, mid_channels, 1, eps=eps, momentum=momentum)
        self.poolings = nn.ModuleList([
            nn.MaxPool2d(kernel_size=ks, stride=1, padding=ks // 2) for ks in kernel_sizes
        ])
        self.conv2 = ConvBNAct(mid_channels * (len(kernel_sizes) + 1), out_channels, 1, eps=eps, momentum=momentum)

    def forward(self, x):
        x = self.conv1(x)
        with torch.cuda.amp.autocast(enabled=False):
            x32 = x.float()
            x = torch.cat([x32] + [pool(x32) for pool in self.poolings], dim=1)
        x = x.to(dtype=self.conv2.conv.weight.dtype)
        return self.conv2(x)


class CSPNeXtBlock(nn.Module):
    def __init__(self, in_channels, out_channels, expansion=0.5, add_identity=True, kernel_size=5,
                 eps=1e-3, momentum=0.01):
        super().__init__()
        hidden_channels = int(out_channels * expansion)
        self.conv1 = ConvBNAct(in_channels, hidden_channels, 3, eps=eps, momentum=momentum)
        self.conv2 = nn.Sequential(
            ConvBNAct(hidden_channels, hidden_channels, kernel_size, groups=hidden_channels, eps=eps, momentum=momentum),
            ConvBNAct(hidden_channels, out_channels, 1, eps=eps, momentum=momentum),
        )
        self.add_identity = add_identity and in_channels == out_channels

    def forward(self, x):
        y = self.conv1(x)
        y = self.conv2(y)
        return y + x if self.add_identity else y


class CSPLayer(nn.Module):
    def __init__(self, in_channels, out_channels, expand_ratio=0.5, num_blocks=1,
                 add_identity=True, channel_attention=True, eps=1e-3, momentum=0.01):
        super().__init__()
        mid_channels = int(out_channels * expand_ratio)
        self.main_conv = ConvBNAct(in_channels, mid_channels, 1, eps=eps, momentum=momentum)
        self.short_conv = ConvBNAct(in_channels, mid_channels, 1, eps=eps, momentum=momentum)
        self.final_conv = ConvBNAct(2 * mid_channels, out_channels, 1, eps=eps, momentum=momentum)
        self.blocks = nn.Sequential(*[
            CSPNeXtBlock(mid_channels, mid_channels, expansion=1.0, add_identity=add_identity, eps=eps, momentum=momentum)
            for _ in range(num_blocks)
        ])
        self.attention = ChannelAttention(2 * mid_channels) if channel_attention else nn.Identity()

    def forward(self, x):
        x_short = self.short_conv(x)
        x_main = self.main_conv(x)
        x_main = self.blocks(x_main)
        x = torch.cat((x_main, x_short), dim=1)
        x = self.attention(x)
        return self.final_conv(x)


class CSPNeXtBackboneFromRTMDet(nn.Module):
    arch_settings = {
        'P5': [
            [64, 128, 3, True, False],
            [128, 256, 6, True, False],
            [256, 512, 6, True, False],
            [512, 1024, 3, False, True],
        ]
    }

    def __init__(self, arch='P5', deepen_factor=1.0, widen_factor=1.0,
                 expand_ratio=0.5, channel_attention=True,
                 out_indices=(2, 3, 4), eps=1e-3, momentum=0.01):
        super().__init__()
        arch_setting = self.arch_settings[arch]
        self.out_indices = out_indices
        c0 = int(arch_setting[0][0] * widen_factor)
        self.stem = nn.Sequential(
            ConvBNAct(3, c0 // 2, 3, stride=2, eps=eps, momentum=momentum),
            ConvBNAct(c0 // 2, c0 // 2, 3, stride=1, eps=eps, momentum=momentum),
            ConvBNAct(c0 // 2, c0, 3, stride=1, eps=eps, momentum=momentum),
        )
        self.layers = ['stem']
        for i, (in_c, out_c, num_blocks, add_identity, use_spp) in enumerate(arch_setting):
            in_c = int(in_c * widen_factor)
            out_c = int(out_c * widen_factor)
            num_blocks = max(round(num_blocks * deepen_factor), 1)
            stage = [ConvBNAct(in_c, out_c, 3, stride=2, eps=eps, momentum=momentum)]
            if use_spp:
                stage.append(SPPBottleneck(out_c, out_c, eps=eps, momentum=momentum))
            stage.append(CSPLayer(
                out_c, out_c,
                expand_ratio=expand_ratio,
                num_blocks=num_blocks,
                add_identity=add_identity,
                channel_attention=channel_attention,
                eps=eps,
                momentum=momentum,
            ))
            self.add_module(f'stage{i + 1}', nn.Sequential(*stage))
            self.layers.append(f'stage{i + 1}')

    def forward(self, x):
        outs = []
        for i, layer_name in enumerate(self.layers):
            layer = getattr(self, layer_name)
            x = layer(x)
            if i in self.out_indices:
                outs.append(x)
        return tuple(outs)


def extract_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if 'state_dict' in ckpt:
            return ckpt['state_dict']
        if 'model' in ckpt and isinstance(ckpt['model'], dict):
            return ckpt['model']
    return ckpt


def load_rtmdet_pretrained_backbone(backbone: nn.Module, ckpt_path: Path):
    if not Path(ckpt_path).exists():
        raise FileNotFoundError(ckpt_path)
    ckpt = torch.load(ckpt_path, map_location='cpu')
    state_dict = extract_state_dict(ckpt)
    filtered = {}
    for k, v in state_dict.items():
        if k.startswith('backbone.'):
            filtered[k[len('backbone.'):]] = v

    missing, unexpected = backbone.load_state_dict(filtered, strict=False)
    loaded_keys = set(filtered.keys()) - set(unexpected)
    summary = {
        'checkpoint': str(ckpt_path),
        'raw_keys': len(state_dict),
        'backbone_candidate_keys': len(filtered),
        'loaded_keys': len(loaded_keys),
        'missing_keys': len(missing),
        'unexpected_keys': len(unexpected),
    }
    print(json.dumps(summary, indent=2))
    if len(missing):
        print('sample missing:', missing[:20])
    if len(unexpected):
        print('sample unexpected:', unexpected[:20])
    return summary


In [ ]:
class _UNetBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class SmallUNet(nn.Module):
    def __init__(self, in_c, base_c=32, out_c=1):
        super().__init__()
        self.in_proj = nn.Conv2d(in_c, base_c, 1)
        c = [base_c, base_c * 2, base_c * 4, base_c * 8]
        self.enc1 = _UNetBlock(c[0], c[0])
        self.enc2 = _UNetBlock(c[0], c[1])
        self.enc3 = _UNetBlock(c[1], c[2])
        self.bot = _UNetBlock(c[2], c[3])
        self.dec3 = _UNetBlock(c[3] + c[2], c[2])
        self.dec2 = _UNetBlock(c[2] + c[1], c[1])
        self.dec1 = _UNetBlock(c[1] + c[0], c[0])
        self.out = nn.Conv2d(c[0], out_c, 1)
        self.pool = nn.MaxPool2d(2)

    def _up(self, x, ref):
        return F.interpolate(x, size=ref.shape[-2:], mode='bilinear', align_corners=False)

    def forward(self, x):
        x = self.in_proj(x)
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bot(self.pool(e3))
        d3 = self.dec3(torch.cat([self._up(b, e3), e3], 1))
        d2 = self.dec2(torch.cat([self._up(d3, e2), e2], 1))
        d1 = self.dec1(torch.cat([self._up(d2, e1), e1], 1))
        return self.out(d1)


class MultiCamBEVv4RTMDet(nn.Module):
    def __init__(self, num_rover_classes: int,
                 rover_emb_dim: int = 8,
                 rover_cond_dim: int = 8,
                 n_cameras: int = 4,
                 freeze_backbone: bool = False,
                 pretrained_backbone_path: str | None = None):
        super().__init__()
        self.n_cameras = n_cameras
        self.bev_h, self.bev_w = BEV_H, BEV_W
        self.x_range, self.y_range = X_RANGE, Y_RANGE
        self.z_levels = Z_LEVELS
        self.rover_cond_dim = rover_cond_dim

        self.backbone = CSPNeXtBackboneFromRTMDet(
            arch='P5',
            deepen_factor=1.0,
            widen_factor=1.0,
            expand_ratio=0.5,
            channel_attention=True,
            out_indices=(2, 3, 4),
            eps=1e-3,
            momentum=0.01,
        )
        self.backbone_load_summary = None
        if pretrained_backbone_path:
            self.backbone_load_summary = load_rtmdet_pretrained_backbone(self.backbone, Path(pretrained_backbone_path))
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        self.backbone_proj = nn.Conv2d(1024, 128, 1)
        self.feat_proj = nn.Conv2d(128, 64, 1)
        self.register_buffer('ego_voxels', self._make_ego_voxels(), persistent=False)

        self.rover_embed = nn.Embedding(num_rover_classes, rover_emb_dim)
        nn.init.normal_(self.rover_embed.weight, std=0.02)
        self.rover_mlp = nn.Sequential(
            nn.Linear(rover_emb_dim, 16),
            nn.ReLU(inplace=True),
            nn.Linear(16, rover_cond_dim),
            nn.ReLU(inplace=True),
        )

        in_c = 64 * len(self.z_levels) + rover_cond_dim
        self.bev_decoder = SmallUNet(in_c=in_c, base_c=32, out_c=1)

    def _make_ego_voxels(self):
        xs = torch.linspace(self.x_range[0] + BEV_RES / 2, self.x_range[1] - BEV_RES / 2, self.bev_h)
        ys = torch.linspace(self.y_range[0] + BEV_RES / 2, self.y_range[1] - BEV_RES / 2, self.bev_w)
        zs = torch.tensor(self.z_levels, dtype=torch.float32)
        Z, X, Y = torch.meshgrid(zs, xs, ys, indexing='ij')
        ones = torch.ones_like(X)
        return torch.stack([X, Y, Z, ones], dim=-1)

    def forward(self, images, intrinsics, car2cams, rover_ids):
        B, N, C_, Hi, Wi = images.shape
        assert N == self.n_cameras

        feats = self.backbone(images.reshape(B * N, C_, Hi, Wi))
        feat = feats[-1]
        feat = self.backbone_proj(feat)
        feat = self.feat_proj(feat)
        Hf, Wf = feat.shape[-2:]
        feat = feat.reshape(B, N, 64, Hf, Wf)

        Z, H, W, _ = self.ego_voxels.shape
        V = Z * H * W
        voxels = self.ego_voxels.reshape(-1, 4).unsqueeze(0).unsqueeze(0).expand(B, N, -1, -1)

        p_cam = torch.einsum('bnij,bnvj->bniv', car2cams, voxels)
        p_cam_3d = p_cam[:, :, :3]
        uv_homog = torch.einsum('bnij,bnjv->bniv', intrinsics, p_cam_3d)
        z = uv_homog[:, :, 2]
        u = uv_homog[:, :, 0] / z.clamp(min=1e-3)
        v = uv_homog[:, :, 1] / z.clamp(min=1e-3)

        u_n = 2.0 * u / Wi - 1.0
        v_n = 2.0 * v / Hi - 1.0
        valid = (z > 0.1) & (u_n.abs() <= 1.0) & (v_n.abs() <= 1.0)

        grid = torch.stack([u_n, v_n], dim=-1).reshape(B * N, V, 1, 2)
        sampled = F.grid_sample(
            feat.reshape(B * N, 64, Hf, Wf),
            grid,
            mode='bilinear',
            padding_mode='zeros',
            align_corners=False,
        )
        sampled = sampled.squeeze(-1).reshape(B, N, 64, V)

        valid_f = valid.float().unsqueeze(2)
        sampled = sampled * valid_f
        agg = sampled.sum(dim=1) / valid_f.sum(dim=1).clamp(min=1.0)
        agg = agg.reshape(B, 64, Z, H, W).reshape(B, 64 * Z, H, W)

        rover_feat = self.rover_mlp(self.rover_embed(rover_ids)).view(B, self.rover_cond_dim, 1, 1)
        rover_map = rover_feat.expand(-1, -1, H, W)
        agg = torch.cat([agg, rover_map], dim=1)
        return self.bev_decoder(agg)


In [ ]:
def _lovasz_grad(gt_sorted: torch.Tensor) -> torch.Tensor:
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1.0 - gt_sorted).float().cumsum(0)
    jaccard = 1.0 - intersection / union
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard


def lovasz_hinge_flat(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    if labels.numel() == 0:
        return logits.sum() * 0.0
    signs = 2.0 * labels.float() - 1.0
    errors = 1.0 - logits * signs
    errors_sorted, perm = torch.sort(errors, dim=0, descending=True)
    gt_sorted = labels[perm]
    grad = _lovasz_grad(gt_sorted)
    return torch.dot(F.relu(errors_sorted), grad)


class CompoundLossV2(nn.Module):
    def __init__(self, pos_weight: float = 5.0,
                 weight_bce: float = 0.5,
                 weight_dice: float = 0.3,
                 weight_lovasz: float = 0.2,
                 ignore_value: int = 255):
        super().__init__()
        self.register_buffer('pos_weight', torch.tensor([pos_weight]))
        self.weight_bce = weight_bce
        self.weight_dice = weight_dice
        self.weight_lovasz = weight_lovasz
        self.ignore_value = ignore_value

    def forward(self, logits: torch.Tensor, gt: torch.Tensor):
        valid = (gt != self.ignore_value)
        gt_f = (gt == 1).float()
        bce_per = F.binary_cross_entropy_with_logits(logits, gt_f, pos_weight=self.pos_weight, reduction='none')
        bce = (bce_per * valid.float()).sum() / valid.float().sum().clamp(min=1.0)
        prob = torch.sigmoid(logits) * valid.float()
        gt_d = gt_f * valid.float()
        inter = (prob * gt_d).sum(dim=(1, 2, 3))
        denom = prob.sum(dim=(1, 2, 3)) + gt_d.sum(dim=(1, 2, 3))
        dice = (1.0 - (2 * inter + 1) / (denom + 1)).mean()
        lov_logits = logits[valid]
        lov_gt = gt_f[valid]
        lov = lovasz_hinge_flat(lov_logits, lov_gt) if lov_logits.numel() > 0 else logits.sum() * 0.0
        total = self.weight_bce * bce + self.weight_dice * dice + self.weight_lovasz * lov
        parts = {'bce': bce.item(), 'dice': dice.item(), 'lovasz': lov.item()}
        return total, parts


@torch.no_grad()
def iou_binary_batch(logits: torch.Tensor, gt: torch.Tensor, threshold: float = 0.5, ignore_value: int = 255):
    probs = torch.sigmoid(logits)
    pred = probs > threshold
    valid = gt != ignore_value
    gt_b = (gt == 1) & valid
    pred = pred & valid
    inter = (pred & gt_b).sum().item()
    union = (pred | gt_b).sum().item()
    return inter, union


def unwrap_model(model):
    return model.module if hasattr(model, 'module') else model


@torch.no_grad()
def update_ema(ema_model, model, decay):
    src = unwrap_model(model)
    ema_params = dict(ema_model.named_parameters())
    src_params = dict(src.named_parameters())
    for name, param in src_params.items():
        ema_params[name].mul_(decay).add_(param.data, alpha=1.0 - decay)
    ema_buffers = dict(ema_model.named_buffers())
    src_buffers = dict(src.named_buffers())
    for name, buf in src_buffers.items():
        ema_buffers[name].copy_(buf)


def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


@torch.inference_mode()
def evaluate_iou(model, loader, threshold=0.5, desc='val@0.5'):
    model.eval()
    inter, union = 0, 0
    pbar = tqdm(loader, desc=desc, leave=False)
    for batch in pbar:
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)
        gt = batch['gt'].to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = model(images, intr, c2c, rover_id).float()
        i, u = iou_binary_batch(logits, gt, threshold=threshold)
        inter += i
        union += u
        pbar.set_postfix(iou=f'{inter / max(union, 1):.4f}')
    return inter / max(union, 1)


@torch.inference_mode()
def streaming_threshold_sweep(model, loader, thresholds):
    model.eval()
    inter = torch.zeros(len(thresholds), dtype=torch.float64)
    union = torch.zeros(len(thresholds), dtype=torch.float64)
    for batch in tqdm(loader, desc='threshold sweep'):
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)
        gt = batch['gt']
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = model(images, intr, c2c, rover_id).float()
        probs = torch.sigmoid(logits).cpu()
        valid = gt != 255
        gt_b = ((gt == 1) & valid).float()
        for ti, t in enumerate(thresholds):
            pred = ((probs > t) & valid).float()
            inter[ti] += (pred * gt_b).sum().item()
            union[ti] += ((pred + gt_b).clamp(0, 1)).sum().item()
    return {float(t): float(inter[i] / max(union[i].item(), 1.0)) for i, t in enumerate(thresholds)}



In [ ]:
ds_train_warm = BEVDatasetV4RTMDet(train_info, mode='train', img_hw=cfg['img_hw'], aug=False, rover_vocab=rover_vocab)
ds_train_aug = BEVDatasetV4RTMDet(train_info, mode='train', img_hw=cfg['img_hw'], aug=True, rover_vocab=rover_vocab)
ds_val = BEVDatasetV4RTMDet(val_info, mode='val', img_hw=cfg['img_hw'], aug=False, rover_vocab=rover_vocab)

loader_warm = DataLoader(
    ds_train_warm,
    batch_size=cfg['batch_size'],
    shuffle=True,
    num_workers=cfg['num_workers'],
    pin_memory=(device.type == 'cuda'),
    drop_last=True,
)
loader_train = DataLoader(
    ds_train_aug,
    batch_size=cfg['batch_size'],
    shuffle=True,
    num_workers=cfg['num_workers'],
    pin_memory=(device.type == 'cuda'),
    drop_last=True,
)
loader_val = DataLoader(
    ds_val,
    batch_size=cfg['val_batch_size'],
    shuffle=False,
    num_workers=cfg['val_num_workers'],
    pin_memory=(device.type == 'cuda'),
)

print('loader_warm batch_size:', loader_warm.batch_size, '| workers:', loader_warm.num_workers)
print('loader_train batch_size:', loader_train.batch_size, '| workers:', loader_train.num_workers)
print('loader_val batch_size:', loader_val.batch_size, '| workers:', loader_val.num_workers)

sample = ds_train_warm[0]
for k, v in sample.items():
    if isinstance(v, torch.Tensor):
        print(k, tuple(v.shape), v.dtype)
    else:
        print(k, type(v), v)



In [ ]:
sanity_model = MultiCamBEVv4RTMDet(
    num_rover_classes=len(rover_vocab),
    rover_emb_dim=cfg['rover_emb_dim'],
    rover_cond_dim=cfg['rover_cond_dim'],
    pretrained_backbone_path=str(RTMDET_PRETRAIN_PATH),
).to(device)

with torch.no_grad():
    batch = next(iter(loader_warm))
    images = batch['images'].to(device)
    intr = batch['intrinsics'].to(device)
    c2c = batch['car2cams'].to(device)
    rover_id = batch['rover_id'].to(device)
    out = sanity_model(images, intr, c2c, rover_id)
    print('sanity output shape:', tuple(out.shape))

del sanity_model, out, batch, images, intr, c2c, rover_id
if device.type == 'cuda':
    torch.cuda.empty_cache()


In [ ]:
base_model = MultiCamBEVv4RTMDet(
    num_rover_classes=len(rover_vocab),
    rover_emb_dim=cfg['rover_emb_dim'],
    rover_cond_dim=cfg['rover_cond_dim'],
    pretrained_backbone_path=str(RTMDET_PRETRAIN_PATH),
).to(device)

if cfg['use_dp_if_available'] and device.type == 'cuda' and torch.cuda.device_count() > 1:
    print('Wrapping model with DataParallel on', torch.cuda.device_count(), 'GPUs')
    model = nn.DataParallel(base_model)
else:
    model = base_model

core_model = unwrap_model(model)
backbone_params, embed_params, other_params = [], [], []
for name, p in core_model.named_parameters():
    if not p.requires_grad:
        continue
    if name.startswith('backbone.') or name.startswith('backbone_proj.'):
        backbone_params.append(p)
    elif name.startswith('rover_embed.'):
        embed_params.append(p)
    else:
        other_params.append(p)

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': cfg['lr_backbone'], 'weight_decay': cfg['weight_decay']},
    {'params': other_params, 'lr': cfg['lr_head'], 'weight_decay': cfg['weight_decay']},
    {'params': embed_params, 'lr': cfg['lr_head'], 'weight_decay': cfg['embed_weight_decay']},
])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['epochs'], eta_min=1e-6)
loss_fn = CompoundLossV2(pos_weight=cfg['pos_weight']).to(device)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

ema_model = copy.deepcopy(core_model).to(device).eval()
for p in ema_model.parameters():
    p.requires_grad = False

print('trainable params M:', sum(p.numel() for p in core_model.parameters() if p.requires_grad) / 1e6)


In [ ]:
log = []
best_iou = -1.0
best_ema_iou = -1.0
start = time.time()

for epoch in range(cfg['epochs']):
    model.train()
    loader = loader_warm if epoch < cfg['warmup_epochs'] else loader_train
    phase = 'WARM' if epoch < cfg['warmup_epochs'] else 'AUG'

    losses = []
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(loader, desc=f'ep{epoch:02d} [{phase}]')

    for step, batch in enumerate(pbar):
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)
        gt = batch['gt'].to(device, non_blocking=True)

        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = model(images, intr, c2c, rover_id)
            loss, parts = loss_fn(logits, gt)
            loss = loss / cfg['grad_accum']

        scaler.scale(loss).backward()

        if (step + 1) % cfg['grad_accum'] == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            update_ema(ema_model, model, cfg['ema_decay'])

        losses.append(loss.item() * cfg['grad_accum'])
        if step % 20 == 0:
            pbar.set_postfix(loss=f'{np.mean(losses[-50:]):.3f}', bce=f"{parts['bce']:.2f}")

    if len(loader) % cfg['grad_accum'] != 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        update_ema(ema_model, model, cfg['ema_decay'])

    scheduler.step()

    cleanup_cuda()
    print('start val model @0.5')
    val_iou_05 = evaluate_iou(model, loader_val, threshold=0.5, desc=f'val model ep{epoch:02d}')

    cleanup_cuda()
    print('start val ema @0.5')
    val_iou_05_ema = evaluate_iou(ema_model, loader_val, threshold=0.5, desc=f'val ema ep{epoch:02d}')
    cleanup_cuda()

    elapsed = (time.time() - start) / 60

    row = {
        'epoch': epoch,
        'loss': float(np.mean(losses)),
        'val_iou_05': float(val_iou_05),
        'val_iou_05_ema': float(val_iou_05_ema),
        'minutes': float(elapsed),
    }

    print(
        f"ep{epoch:02d} | loss {np.mean(losses):.3f} | "
        f"val@0.5 {val_iou_05:.4f} | ema@0.5 {val_iou_05_ema:.4f} | "
        f"{elapsed:.1f}m"
    )

    if val_iou_05 > best_iou:
        best_iou = val_iou_05
        torch.save({
            'model_type': 'v4_rtmdet_pretrain',
            'model': unwrap_model(model).state_dict(),
            'epoch': epoch,
            'best_iou': float(val_iou_05),
            'best_t': 0.5,
            'rover_vocab': rover_vocab,
            'cfg': cfg,
        }, RUN_DIR / 'best.pt')
        print('  new best model:', val_iou_05)

    if val_iou_05_ema > best_ema_iou:
        best_ema_iou = val_iou_05_ema
        torch.save({
            'model_type': 'v4_rtmdet_pretrain',
            'model': unwrap_model(model).state_dict(),
            'ema': ema_model.state_dict(),
            'epoch': epoch,
            'best_ema_iou': float(val_iou_05_ema),
            'best_t_ema': 0.5,
            'rover_vocab': rover_vocab,
            'cfg': cfg,
        }, RUN_DIR / 'ema_best.pt')
        print('  new best ema:', val_iou_05_ema)

    log.append(row)
    pd.DataFrame(log).to_csv(RUN_DIR / 'log.csv', index=False)

    torch.save({
        'model_type': 'v4_rtmdet_pretrain',
        'model': unwrap_model(model).state_dict(),
        'ema': ema_model.state_dict(),
        'epoch': epoch,
        'rover_vocab': rover_vocab,
        'cfg': cfg,
    }, RUN_DIR / 'last.pt')



In [ ]:
ckpt = torch.load(RUN_DIR / 'ema_best.pt', map_location=device) if (RUN_DIR / 'ema_best.pt').exists() else torch.load(RUN_DIR / 'best.pt', map_location=device)
model_eval = MultiCamBEVv4RTMDet(
    num_rover_classes=len(rover_vocab),
    rover_emb_dim=cfg['rover_emb_dim'],
    rover_cond_dim=cfg['rover_cond_dim'],
    pretrained_backbone_path=str(RTMDET_PRETRAIN_PATH),
).to(device)
state = ckpt['ema'] if 'ema' in ckpt else ckpt['model']
model_eval.load_state_dict(state, strict=False)

thresholds = [round(x, 2) for x in np.arange(0.20, 0.81, 0.02)]
iou_by_t = streaming_threshold_sweep(model_eval, loader_val, thresholds)
best_t, best_iou = max(iou_by_t.items(), key=lambda kv: kv[1])
print('best threshold:', best_t, 'best_iou:', best_iou)
for t, iou in iou_by_t.items():
    marker = ' <- best' if t == best_t else ''
    print(f't={t:.2f}  iou={iou:.4f}{marker}')


## Notes

- The notebook uses only the RTMDet image backbone pretrain, not the full BEVFusion stack.
- No new image augmentations are added in this first version.
- The cleaning pipeline, rover embeddings, grouped test-matched split, EMA, and threshold sweep stay inside one notebook.
- All caches and checkpoints are written only into `RUN_DIR`.
